In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable

In [0]:
silver_df=spark.read.table(
    "hospital_appointments.silver.silver_data"
)

In [0]:
%sql
CREATE  or replace TABLE hospital_appointments.gold.dim_patient
(
    patient_key BIGINT GENERATED ALWAYS AS IDENTITY,
    patient_id STRING,
    patient_name STRING,
    patient_age INT,
    patient_gender STRING
);


In [0]:
%sql
MERGE INTO hospital_appointments.gold.dim_patient target
USING
(
    SELECT
        patient_id,
        patient_name,
        patient_age,
        patient_gender
    FROM
    (
        SELECT *,
               ROW_NUMBER() OVER
               (
                   PARTITION BY patient_id
                   ORDER BY appointment_date DESC
               ) rn
        FROM hospital_appointments.silver.silver_data
    )
    WHERE rn = 1
) source
ON target.patient_id = source.patient_id

WHEN MATCHED THEN
UPDATE SET
    target.patient_name = source.patient_name,
    target.patient_age = source.patient_age,
    target.patient_gender = source.patient_gender

WHEN NOT MATCHED THEN
INSERT
(
    patient_id,
    patient_name,
    patient_age,
    patient_gender
)
VALUES
(
    source.patient_id,
    source.patient_name,
    source.patient_age,
    source.patient_gender
);

In [0]:
%sql
select * from hospital_appointments.gold.dim_patient

In [0]:
%sql
CREATE OR REPLACE TABLE hospital_appointments.gold.dim_doctor
(
    doctor_key BIGINT GENERATED ALWAYS AS IDENTITY,
    doctor_id STRING,
    doctor_name STRING,
    department STRING
);

In [0]:
%sql
MERGE INTO hospital_appointments.gold.dim_doctor AS target
USING
(
    SELECT
        doctor_id,
        doctor_name,
        department
    FROM
    (
        SELECT *,
               ROW_NUMBER() OVER
               (
                   PARTITION BY doctor_id
                   ORDER BY appointment_date DESC
               ) rn
        FROM hospital_appointments.silver.silver_data
        WHERE doctor_id IS NOT NULL
    )
    WHERE rn = 1
) source

ON target.doctor_id = source.doctor_id

WHEN MATCHED THEN
UPDATE SET
    target.doctor_name = source.doctor_name,
    target.department = source.department

WHEN NOT MATCHED THEN
INSERT
(
    doctor_id,
    doctor_name,
    department
)
VALUES
(
    source.doctor_id,
    source.doctor_name,
    source.department
);

In [0]:
%sql
select * from hospital_appointments.gold.dim_doctor;

In [0]:
%sql
CREATE OR REPLACE TABLE hospital_appointments.gold.dim_date
(
    date_key INT,
    full_date DATE,
    year INT,
    month INT,
    quarter INT,
    day INT
);

In [0]:
%sql
MERGE INTO hospital_appointments.gold.dim_date target
USING
(
    SELECT DISTINCT
        CAST(DATE_FORMAT(appointment_date,'yyyyMMdd') AS INT) AS date_key,
        CAST(appointment_date AS DATE) AS full_date,
        YEAR(appointment_date) AS year,
        MONTH(appointment_date) AS month,
        QUARTER(appointment_date) AS quarter,
        DAY(appointment_date) AS day
    FROM hospital_appointments.silver.silver_data
    WHERE appointment_date IS NOT NULL
) source
ON target.date_key = source.date_key

WHEN NOT MATCHED THEN
INSERT
(
    date_key,
    full_date,
    year,
    month,
    quarter,
    day
)
VALUES
(
    source.date_key,
    source.full_date,
    source.year,
    source.month,
    source.quarter,
    source.day
);

In [0]:
%sql
select * from hospital_appointments.gold.dim_date

In [0]:
%sql
CREATE OR REPLACE TABLE hospital_appointments.gold.dim_city
(
    city_key BIGINT GENERATED ALWAYS AS IDENTITY,
    city_name STRING
);



In [0]:
%sql
MERGE INTO hospital_appointments.gold.dim_city target
USING
(
    SELECT DISTINCT hospital_city AS city_name
FROM hospital_appointments.silver.silver_data
    WHERE hospital_city IS NOT NULL
) source
ON target.city_name = source.city_name

WHEN NOT MATCHED THEN
INSERT
(
    city_name
)
VALUES
(
    source.city_name
);

In [0]:
%sql
select * from hospital_appointments.gold.dim_city

In [0]:
%sql
CREATE OR REPLACE TABLE hospital_appointments.gold.fact_appointments
AS
SELECT
    s.appointment_id,
    p.patient_key,
    d.doctor_key,
    c.city_key,
    dt.date_key,
    s.appointment_date,
    s.appointment_status,
    s.consultation_fee,
    s.test_charge,
    s.appointment_amount

FROM hospital_appointments.silver.silver_data s

LEFT JOIN hospital_appointments.gold.dim_patient p
    ON s.patient_id = p.patient_id

LEFT JOIN hospital_appointments.gold.dim_doctor d
    ON s.doctor_id = d.doctor_id

LEFT JOIN hospital_appointments.gold.dim_city c
    ON s.city = c.city_name

LEFT JOIN hospital_appointments.gold.dim_date dt
    ON s.appointment_date = dt.full_date;

In [0]:
%sql
select * from hospital_appointments.gold.fact_appointments

### Department_Revenue_Summary

In [0]:
%sql
CREATE OR REPLACE TABLE hospital_appointments.gold.department_revenue_summary
USING DELTA
AS
SELECT
    d.department,
    COUNT(f.appointment_id) AS total_appointments,
     SUM(
        CASE
            WHEN f.appointment_status = 'Completed'
            THEN 1
            ELSE 0
        END
    ) AS completed_appointments,

    SUM(
        CASE
            WHEN f.appointment_status = 'Completed'
            THEN f.appointment_amount
            ELSE 0
        END
    ) AS total_revenue,

    ROUND(
        AVG(
            CASE
                WHEN f.appointment_status = 'Completed'
                THEN f.appointment_amount
            END
        ), 2
    ) AS average_revenue

FROM hospital_appointments.gold.fact_appointments f
JOIN hospital_appointments.gold.dim_doctor d
    ON f.doctor_key = d.doctor_key
GROUP BY d.department;

In [0]:
%sql
select * from hospital_appointments.gold.department_revenue_summary

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

#### Doctor_Performance_Summary

In [0]:
%sql
CREATE OR REPLACE TABLE hospital_appointments.gold.doctor_performance_summary

AS
SELECT
    d.doctor_id,
    d.doctor_name,
    d.department,

    COUNT(f.appointment_id) AS total_appointments,

    SUM(CASE
        WHEN f.appointment_status = 'Completed' THEN 1
        ELSE 0
    END) AS completed_appointments,

    SUM(CASE
        WHEN f.appointment_status = 'Cancelled' THEN 1
        ELSE 0
    END) AS cancelled_appointments,

    SUM(CASE
        WHEN f.appointment_status = 'Completed'
        THEN f.appointment_amount
        ELSE 0
    END) AS total_revenue

FROM hospital_appointments.gold.fact_appointments f
JOIN hospital_appointments.gold.dim_doctor d
    ON f.doctor_key = d.doctor_key
GROUP BY
    d.doctor_id,
    d.doctor_name,
    d.department;

In [0]:
%sql
select * from hospital_appointments.gold.doctor_performance_summary


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

### Monthly_Revenue_Summary


In [0]:
%sql
CREATE OR REPLACE TABLE hospital_appointments.gold.monthly_revenue_summary
AS
SELECT
    dt.year,
    dt.month,
    COUNT(f.appointment_id) AS total_appointments,

    SUM(
        CASE
            WHEN f.appointment_status = 'Completed'
            THEN f.appointment_amount
            ELSE 0
        END
    ) AS total_revenue

FROM hospital_appointments.gold.fact_appointments f
JOIN hospital_appointments.gold.dim_date dt
    ON f.date_key = dt.date_key
GROUP BY
    dt.year,
    dt.month;

In [0]:
%sql
select * from hospital_appointments.gold.monthly_revenue_summary

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

### City_Patient_Summary

In [0]:
%sql
CREATE OR REPLACE TABLE hospital_appointments.gold.city_patient_summary
AS
SELECT
    c.city_name,
    COUNT(DISTINCT p.patient_id) AS total_patients,
    COUNT(f.appointment_id) AS total_appointments,

    SUM(
        CASE
            WHEN f.appointment_status = 'Completed'
            THEN f.appointment_amount
            ELSE 0
        END
    ) AS total_revenue

FROM hospital_appointments.gold.fact_appointments f
JOIN hospital_appointments.gold.dim_patient p
    ON f.patient_key = p.patient_key
JOIN hospital_appointments.gold.dim_city c
    ON f.city_key = c.city_key
GROUP BY c.city_name;

In [0]:
%sql
select * from hospital_appointments.gold.city_patient_summary

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.